# Inference notebook

This notebook will try to use gpt-2 and gpt-4 regex and merges to make a decode and encode function and compare with tiktoken encode function.

In [1]:
import tiktoken

gpt2 = tiktoken.get_encoding("gpt2") # gpt2
gpt4 = tiktoken.get_encoding("cl100k_base")  # gpt4

gpt2_regex = gpt2._pat_str
gpt4_regex = gpt4._pat_str

print(f"gpt2 pattern: {gpt2_regex}")
print(f"gpt4 pattern: {gpt4_regex}")

# GPT4 regex is much more complex and a lot more rules than gpt2.

gpt2 pattern: '(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s
gpt4 pattern: '(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s


In [2]:
merges_gpt2 = gpt2._mergeable_ranks
merges_gpt4 = gpt4._mergeable_ranks

print(type(merges_gpt2))

<class 'dict'>


In [3]:
key_list = list(merges_gpt2.keys())
key_list[:100]

[b'!',
 b'"',
 b'#',
 b'$',
 b'%',
 b'&',
 b"'",
 b'(',
 b')',
 b'*',
 b'+',
 b',',
 b'-',
 b'.',
 b'/',
 b'0',
 b'1',
 b'2',
 b'3',
 b'4',
 b'5',
 b'6',
 b'7',
 b'8',
 b'9',
 b':',
 b';',
 b'<',
 b'=',
 b'>',
 b'?',
 b'@',
 b'A',
 b'B',
 b'C',
 b'D',
 b'E',
 b'F',
 b'G',
 b'H',
 b'I',
 b'J',
 b'K',
 b'L',
 b'M',
 b'N',
 b'O',
 b'P',
 b'Q',
 b'R',
 b'S',
 b'T',
 b'U',
 b'V',
 b'W',
 b'X',
 b'Y',
 b'Z',
 b'[',
 b'\\',
 b']',
 b'^',
 b'_',
 b'`',
 b'a',
 b'b',
 b'c',
 b'd',
 b'e',
 b'f',
 b'g',
 b'h',
 b'i',
 b'j',
 b'k',
 b'l',
 b'm',
 b'n',
 b'o',
 b'p',
 b'q',
 b'r',
 b's',
 b't',
 b'u',
 b'v',
 b'w',
 b'x',
 b'y',
 b'z',
 b'{',
 b'|',
 b'}',
 b'~',
 b'\xa1',
 b'\xa2',
 b'\xa3',
 b'\xa4',
 b'\xa5',
 b'\xa6']

In [4]:
val_list = list(merges_gpt2.values())
val_list[:100]

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99]

In [5]:
len(val_list)

50256

In [6]:
val_list_gpt4 = list(merges_gpt4.values())
len(val_list_gpt4)

100256

GPT-4 is basically double of GPT-2 vocab size.

Now, for GPT-2 exposes 2 files, vocab and merges which makes it easier to implement the inference side.

GPT-4 is trickier as only the ranks (vocab) is exposed, and hence, merges have to be constructed by ourselves.

GPT-2 files can be downloaded from https://openaipublic.blob.core.windows.net/gpt-2/models/124M [encoder.json, vocab.bpe]

In [7]:
import os
import json

dir = os.getcwd()
gpt_files = os.path.join(dir, "..", "openai_files")

encoder_path = os.path.join(gpt_files, "encoder.json")
vocab_merges_path = os.path.join(gpt_files, "vocab.bpe")

with open(encoder_path, "r", encoding="utf-8") as f:
    vocab_dictionary = json.load(f)
    
len(vocab_dictionary)

50257

In [8]:
# right now its {byte: index}, we need to reverse this

with open(vocab_merges_path, "r", encoding="utf-8") as f:
    merges_gpt2_from_file = [
        tuple(line.split()) for line in f.read().splitlines()[1:]
    ]
    
print("Vocabulary size: ", len(vocab_dictionary))
print("Number of merges: ", len(merges_gpt2_from_file))
print("First 10 merges: ", merges_gpt2_from_file[:10])

Vocabulary size:  50257
Number of merges:  50000
First 10 merges:  [('Ġ', 't'), ('Ġ', 'a'), ('h', 'e'), ('i', 'n'), ('r', 'e'), ('o', 'n'), ('Ġt', 'he'), ('e', 'r'), ('Ġ', 's'), ('a', 't')]


In [9]:
# gpt2 byte to unicode mapping

def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1))
    bs += list(range(ord("¡"), ord("¬") + 1))
    bs += list(range(ord("®"), ord("ÿ") + 1))
    
    cs = bs[:]
    n = 0
    
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
            
    cs = [chr(c) for c in cs]
    
    return dict(zip(bs, cs))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

In [10]:
print(byte_encoder[97])
print(byte_encoder[32])
print(byte_decoder[byte_encoder[97]])

a
Ġ
97


In [11]:
gpt2_vocab = {idx : token for token, idx in vocab_dictionary.items()}

gpt2_vocab[50256]

'<|endoftext|>'

In [12]:
for i in range(100):
    print(gpt2_vocab[i])

!
"
#
$
%
&
'
(
)
*
+
,
-
.
/
0
1
2
3
4
5
6
7
8
9
:
;
<
=
>
?
@
A
B
C
D
E
F
G
H
I
J
K
L
M
N
O
P
Q
R
S
T
U
V
W
X
Y
Z
[
\
]
^
_
`
a
b
c
d
e
f
g
h
i
j
k
l
m
n
o
p
q
r
s
t
u
v
w
x
y
z
{
|
}
~
¡
¢
£
¤
¥
¦


In [13]:
bpe_ranks_gpt2 = {
    pair: i
    for i, pair in enumerate(merges_gpt2_from_file)
}

print(bpe_ranks_gpt2[merges_gpt2_from_file[0]])

keys = list(bpe_ranks_gpt2.keys())

print(keys[:100])

0
[('Ġ', 't'), ('Ġ', 'a'), ('h', 'e'), ('i', 'n'), ('r', 'e'), ('o', 'n'), ('Ġt', 'he'), ('e', 'r'), ('Ġ', 's'), ('a', 't'), ('Ġ', 'w'), ('Ġ', 'o'), ('e', 'n'), ('Ġ', 'c'), ('i', 't'), ('i', 's'), ('a', 'n'), ('o', 'r'), ('e', 's'), ('Ġ', 'b'), ('e', 'd'), ('Ġ', 'f'), ('in', 'g'), ('Ġ', 'p'), ('o', 'u'), ('Ġa', 'n'), ('a', 'l'), ('a', 'r'), ('Ġt', 'o'), ('Ġ', 'm'), ('Ġo', 'f'), ('Ġ', 'in'), ('Ġ', 'd'), ('Ġ', 'h'), ('Ġan', 'd'), ('i', 'c'), ('a', 's'), ('l', 'e'), ('Ġt', 'h'), ('i', 'on'), ('o', 'm'), ('l', 'l'), ('en', 't'), ('Ġ', 'n'), ('Ġ', 'l'), ('s', 't'), ('Ġ', 're'), ('v', 'e'), ('Ġ', 'e'), ('r', 'o'), ('l', 'y'), ('Ġb', 'e'), ('Ġ', 'g'), ('Ġ', 'T'), ('c', 't'), ('Ġ', 'S'), ('i', 'd'), ('o', 't'), ('Ġ', 'I'), ('u', 't'), ('e', 't'), ('Ġ', 'A'), ('Ġ', 'is'), ('Ġ', 'on'), ('i', 'm'), ('a', 'm'), ('o', 'w'), ('a', 'y'), ('a', 'd'), ('s', 'e'), ('Ġth', 'at'), ('Ġ', 'C'), ('i', 'g'), ('Ġf', 'or'), ('a', 'c'), ('Ġ', 'y'), ('v', 'er'), ('u', 'r'), ('Ġ', 'u'), ('l', 'd'), ('Ġs', 't'), ('

In [14]:
def get_pairs(word):
    
    pairs = set()
    
    prev = word[0]
    
    for current in word[1:]:
        pairs.add((prev, current))
        prev = current
        
    return pairs

word = "hello"
get_pairs(word)

{('e', 'l'), ('h', 'e'), ('l', 'l'), ('l', 'o')}

In [24]:
def bpe_merge(token):
    
    word = tuple(token)
    pairs = get_pairs(word)
    
    if not pairs:
        return token # whitespace or a unique sequence
    
    while True:
        bigram = min(
            pairs, key=lambda pair: bpe_ranks_gpt2.get(pair, float("inf"))
        )
        
        # here we are finding the pair of chars that appear the earliest in the merge ranks to replace with another new token
        
        if bigram not in bpe_ranks_gpt2:
            break
        
        first, second = bigram
        new_word = []
        i = 0
        
        while i < len(word):
            
            try:
                j = word.index(first, i)
            except ValueError:
                new_word.extend(word[i:])
                break
        
            new_word.extend(word[i:j])
            i = j
            
            # basically saving time and eliminating parts of word before the pair appears
            
            if (
                word[i] == first
                and i + 1 < len(word)
                and word[i + 1] == second
            ):
                new_word.append(first + second)
                i += 2
                
            else:
                new_word.append(word[i])
                i += 1
                
        word = tuple(new_word)
        
        if len(word) == 1:
            break
        
        pairs = get_pairs(word)
        
    return " ".join(word)

bpe_merge("hello")
bpe_merge(" hello")

'  hello'

In [16]:
import regex

pattern = regex.compile(gpt2_regex)

text = "Hello world! I'm 21 😀"
chunks = pattern.findall(text)
chunks

['Hello', ' world', '!', ' I', "'m", ' 21', ' 😀']

In [25]:
def encode(text):
    ids = []
    
    for token in pattern.findall(text):
        
        token_bytes = token.encode("utf-8")
        
        token_translated = "".join(
            byte_encoder[b] for b in token_bytes
        )
        
        bpe_tokens = bpe_merge(token_translated).split(" ")

        ids.extend(
            vocab_dictionary[token] for token in bpe_tokens
        )
        
    return ids

text = "Hello world!"
ids = encode(text)
ids

[15496, 995, 0]

In [26]:
def decode(ids):
    text = "".join(gpt2_vocab[idx] for idx in ids)
    
    text = bytes(
        byte_decoder[c] for c in text
    )
    
    return text.decode("utf-8")

In [27]:
text = "Hello world! 😀"

ids = encode(text)

print(ids)
print(decode(ids))


[15496, 995, 0, 30325, 222]
Hello world! 😀


In [28]:
text = "Hello world! 😀"

mine = encode(text)
theirs = gpt2.encode(text)

print("Mine:   ", mine)
print("tiktoken:", theirs)

print("Match:", mine == theirs)

Mine:    [15496, 995, 0, 30325, 222]
tiktoken: [15496, 995, 0, 30325, 222]
Match: True


In [29]:
tests = [
    "Hello world!",
    "The quick brown fox jumps over the lazy dog.",
    "I love Weeknd.",
    "1234567890",
    "😀🔥🚀",
    "भारत में हिंदी",
    "def hello_world():\n    print('hello')",
    "Hello     world\n\nThis is a test.",
]

for text in tests:
    mine = encode(text)
    gpt2_tiktoken = gpt2.encode(text)
    
    print(mine == gpt2_tiktoken, repr(text))
    
    print(decode(mine) == gpt2.decode(gpt2_tiktoken))

True 'Hello world!'
True
True 'The quick brown fox jumps over the lazy dog.'
True
True 'I love Weeknd.'
True
True '1234567890'
True
True '😀🔥🚀'
True
True 'भारत में हिंदी'
True
True "def hello_world():\n    print('hello')"
True
True 'Hello     world\n\nThis is a test.'
True
